In [0]:
# --- 1. SET THE CATALOG CONTEXT ---
# This matches the name in your Catalog Explorer
catalog_name = "healthcare_p360" 

# This command "opens" the catalog so the schemas can be seen
spark.sql(f"USE CATALOG {catalog_name}")

# --- 2. DEFINE THE DATA SOURCE ---
# This points to the Volume where your CSVs live
volume_path = f"/Volumes/{catalog_name}/bronze/raw_files/"

# --- 3. INGESTION FUNCTION ---
def ingest_to_bronze(file_name, table_name):
    print(f"Reading {file_name} from {volume_path}...")
    
    try:
        # 1. Read the CSV from the Volume
        df = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(f"{volume_path}{file_name}"))
        
        # 2. Write it as a Delta Table in the Bronze Schema
        full_table_path = f"{catalog_name}.bronze.{table_name}"
        
        (df.write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(full_table_path))
        
        print(f"✅ Successfully created: {full_table_path}")
        
    except Exception as e:
        print(f"❌ Error: {e}")

# --- 4. EXECUTE ---
ingest_to_bronze("patient.csv", "patients")
ingest_to_bronze("lab_result.csv", "labs")
ingest_to_bronze("diagnoses.csv", "diagnoses")